# S-Fig 1 — K-Aggregation

AUROC vs K (number of windows aggregated), at representative context lengths.  
**Source**: analysis.csv (all k values)  
**Tasks**: all tasks

In [ ]:
import sys
from pathlib import Path

# ── Workspace root (parent of NSRR-tools/) ────────────────────────────────────
WORKSPACE_ROOT = Path("../../../../..").resolve()   # adjust if notebook depth differs
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path
sys.path.insert(0, str(Path(".").resolve()))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib
matplotlib.use("Agg")   # comment out in Jupyter to get inline plots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()
print("Setup OK — workspace root:", WORKSPACE_ROOT)

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD     = "transformer"
TASKS    = ALL_TASKS
CONTEXTS = ["40m", "120m", "240m"]   # show these context lengths per panel
HEADS    = ["lstm", "transformer"]
N_COLS   = 4
N_ROWS   = (len(TASKS) + N_COLS - 1) // N_COLS

# Load full analysis.csv (all k values, not just k='all')
import pandas as pd
from pathlib import Path
_ana_path = WORKSPACE_ROOT / "final_results" / "phase0_v3" / "collected" / "analysis.csv"
df_all_k = pd.read_csv(_ana_path)
df_all_k["context_length_min"] = df_all_k["context_length"].map(
    lambda s: {"30s": 0.5, "10m": 10.0, "40m": 40.0,
               "80m": 80.0, "120m": 120.0, "240m": 240.0}.get(str(s).strip())
)
print("Loaded:", df_all_k.shape, "rows")

In [ ]:
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(FULL_W, N_ROWS * 2.0))
axes_flat = axes.flatten()

for i, (ax, task) in enumerate(zip(axes_flat, TASKS)):
    panels.k_agg_panel(ax, df_all_k, task, heads=HEADS, contexts=CONTEXTS)
    ax.set_title(TASK_LABEL.get(task, task), fontsize=8)
    add_panel_label(ax, f"({chr(97+i)})")

for ax in axes_flat[len(TASKS):]:
    ax.set_visible(False)

fig.tight_layout(h_pad=1.5, w_pad=1.0)
save_figure(fig, FINAL_OUT, "sfig1_k_aggregation")
plt.show()